In [5]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from lightning import pytorch as pl
from rdkit import Chem
from chemprop import data, featurizers, models, nn, utils




In [6]:
# ==========================================
# 1. SETUP, CONFIGURATION & DATA CLEANING
# ==========================================
data_dir = Path.cwd()
smiles_col = "smiles"   
target_col = "activity" 
num_workers = 4         

print("Loading and cleaning datasets...")

# --- PRE-TRAINING DATA ---
# Dataset 1: Has 'SMILE' and 'MIC'
df_1 = pd.read_excel(data_dir / "Dataset_1.xlsx")
df_1 = df_1.rename(columns={"SMILE": "smiles"})
# CLEANING FIX: Strip out '>' and '<' symbols and convert to numeric
df_1["MIC"] = pd.to_numeric(df_1["MIC"].astype(str).str.replace('>', '').str.replace('<', '').str.strip(), errors='coerce')
df_1["activity"] = (df_1["MIC"] <= 16).astype(float)
df_1 = df_1[["smiles", "activity"]]

# Dataset 2: Has no headers. 
df_2 = pd.read_excel(data_dir / "Dataset_2.xlsx", header=None)
df_2 = df_2.rename(columns={1: "smiles", 2: "MIC"})
# CLEANING FIX: Strip out '>' and '<' symbols and convert to numeric
df_2["MIC"] = pd.to_numeric(df_2["MIC"].astype(str).str.replace('>', '').str.replace('<', '').str.strip(), errors='coerce')
df_2["activity"] = (df_2["MIC"] <= 16).astype(float)
df_2 = df_2[["smiles", "activity"]]

# Dataset 3: Has PUBCHEM columns
df_3 = pd.read_excel(data_dir / "Dataset_3.xlsx")
df_3 = df_3.rename(columns={"PUBCHEM_EXT_DATASOURCE_SMILES": "smiles"})
df_3["activity"] = df_3["PUBCHEM_ACTIVITY_OUTCOME"].map({"Active": 1.0, "Inactive": 0.0})
df_3 = df_3[["smiles", "activity"]]

# Combine all pre-training data cleanly and drop any rows with missing values
df_pre = pd.concat([df_1, df_2, df_3], ignore_index=True).dropna()


# --- FINE-TUNING DATA ---
# Dataset 4: Has PUBCHEM columns
df_fine = pd.read_excel(data_dir / "Dataset_4.xlsx")
df_fine = df_fine.rename(columns={"PUBCHEM_EXT_DATASOURCE_SMILES": "smiles"})
df_fine["activity"] = df_fine["PUBCHEM_ACTIVITY_OUTCOME"].map({"Active": 1.0, "Inactive": 0.0})
df_fine = df_fine[["smiles", "activity"]].dropna()


# --- TEST SET DATA ---
# Dataset 5: Has PUBCHEM columns
df_test = pd.read_csv(data_dir / "Dataset_5.csv")
df_test = df_test.rename(columns={"PUBCHEM_EXT_DATASOURCE_SMILES": "smiles"})
df_test["activity"] = df_test["PUBCHEM_ACTIVITY_OUTCOME"].map({"Active": 1.0, "Inactive": 0.0})
df_test = df_test[["smiles", "activity"]].dropna()

print(f"Pre-training set size: {len(df_pre)} molecules")
print(f"Fine-tuning set size: {len(df_fine)} molecules")
print(f"Test set size: {len(df_test)} molecules")

Loading and cleaning datasets...
Pre-training set size: 9937 molecules
Fine-tuning set size: 2809 molecules
Test set size: 892 molecules


In [8]:
# Export the cleaned dataframes to CSV for the command-line pipeline
df_pre.to_csv("pre_train.csv", index=False)
df_fine.to_csv("fine_tune.csv", index=False)
df_test.to_csv("test.csv", index=False)

print("CSV files exported successfully!")

CSV files exported successfully!
